## 1. Install library dan import library

In [ ]:
!pip install -q kagglehub huggingface_hub

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from typing import List, Tuple
from huggingface_hub import snapshot_download
import kagglehub

In [ ]:
# Setup Kaggle API di Colab
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## 2. Konfigurasi Utama

In [ ]:
# Format: "username/dataset-slug"
KAGGLE_HANDLE = "afenmarbun/wilddeepfake-46f"

# Dataset Hugging Face
HF_REPO_ID = "xingjunm/WildDeepfake"
HF_ROOT_DIR = "deepfake_in_the_wild"

# Root kerja lokal
BASE_DIR = Path("/kaggle/working/wdf_46f_pipeline")
RAW_CACHE_DIR = BASE_DIR / "raw_cache"
RAW_EXTRACT_DIR = BASE_DIR / "raw_extract"
PROCESSED_DATASET_DIR = BASE_DIR / "wilddeepfake_46f"

# Konfigurasi pembentukan klip
CLIP_LEN = 46  # sesuai proposal
VALID_SUBSETS = {"real_test", "real_train", "fake_test", "fake_train"}

for p in [RAW_CACHE_DIR, RAW_EXTRACT_DIR, PROCESSED_DATASET_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("KAGGLE_HANDLE         :", KAGGLE_HANDLE)
print("HF_REPO_ID            :", HF_REPO_ID)
print("RAW_CACHE_DIR         :", RAW_CACHE_DIR)
print("RAW_EXTRACT_DIR       :", RAW_EXTRACT_DIR)
print("PROCESSED_DATASET_DIR :", PROCESSED_DATASET_DIR)
print("CLIP_LEN              :", CLIP_LEN)

## 3. Fungsi utilitas dasar

In [ ]:
def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def move_contents(src_dir: Path, dst_dir: Path):
    ensure_dir(dst_dir)
    for item in src_dir.iterdir():
        target = dst_dir / item.name
        if target.exists():
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()
        shutil.move(str(item), str(target))


def list_image_files(seq_dir: Path) -> List[Path]:
    allowed_suffixes = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
    files = [p for p in seq_dir.iterdir() if p.is_file() and p.suffix.lower() in allowed_suffixes]

    def sort_key(p: Path):
        stem = p.stem
        try:
            return (0, int(stem))
        except ValueError:
            return (1, stem)

    return sorted(files, key=sort_key)

## 4. Download subset dari Hugging Face

In [ ]:
def download_subset_from_hf(subset_name: str, cache_dir: Path) -> Path:
    if subset_name not in VALID_SUBSETS:
        raise ValueError(f"Subset tidak valid: {subset_name}")

    allow_pattern = f"{HF_ROOT_DIR}/{subset_name}/*"
    print(f"Mengunduh subset: {subset_name}")
    print(f"Allow pattern   : {allow_pattern}")

    local_snapshot_dir = snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        local_dir=str(cache_dir),
        allow_patterns=[allow_pattern],
    )

    subset_path = Path(local_snapshot_dir) / HF_ROOT_DIR / subset_name
    if not subset_path.exists():
        raise FileNotFoundError(f"Subset tidak ditemukan setelah download: {subset_path}")

    print("Download selesai di:", subset_path)
    return subset_path

## 5. Ekstraksi arsip .tar.gz

In [ ]:
def extract_all_archives_in_subset(subset_dir: Path):
    archives = sorted(subset_dir.rglob("*.tar.gz"))
    print(f"Jumlah arsip ditemukan: {len(archives)}")

    if len(archives) == 0:
        print("Tidak ada arsip .tar.gz.")
        return

    for archive_path in archives:
        archive_path = archive_path.resolve()
        parent_dir = archive_path.parent
        print(f"Ekstrak: {archive_path.name} di {parent_dir}")

        cmd = f'cd "{parent_dir}" && tar -xf "{archive_path.name}" && rm "{archive_path.name}"'
        result = subprocess.run(cmd, shell=True, text=True, capture_output=True)

        if result.returncode != 0:
            print("STDERR:")
            print(result.stderr)
            raise RuntimeError(f"Gagal ekstrak arsip: {archive_path}")

    print("Semua arsip selesai diekstrak.")

## 6. Rapikan struktur hasil ekstraksi

In [ ]:
def flatten_inner_class_folder(subset_dir: Path, class_name: str):
    """
    Contoh sebelum:
    real_test/0/real/123/0.png
    menjadi:
    real_test/0/123/0.png
    """
    group_dirs = [p for p in subset_dir.iterdir() if p.is_dir()]

    for group_dir in group_dirs:
        inner_class_dir = group_dir / class_name
        if inner_class_dir.exists() and inner_class_dir.is_dir():
            print(f"Flattening: {inner_class_dir}")
            move_contents(inner_class_dir, group_dir)
            inner_class_dir.rmdir()

## 7. Pembentukan klip 46 frame

In [ ]:
def compute_centered_clip_ranges(num_frames: int, clip_len: int = 46) -> List[Tuple[int, int]]:
    """
    Menghasilkan list rentang [start, end) untuk klip-klip 46 frame.
    Strategi:
    - C = floor(N / L)
    - R = N - C*L
    - buang floor(R/2) di kiri dan ceil(R/2) di kanan
    """
    if num_frames < clip_len:
        return []

    num_clips = num_frames // clip_len
    used_frames = num_clips * clip_len
    remainder = num_frames - used_frames

    discard_left = remainder // 2
    start_idx = discard_left

    ranges = []
    for i in range(num_clips):
        s = start_idx + i * clip_len
        e = s + clip_len
        ranges.append((s, e))

    return ranges

In [ ]:
def build_46f_clips_for_subset(
    extracted_subset_dir: Path,
    processed_subset_dir: Path,
    clip_len: int = 46
):
    """
    Struktur hasil:
    processed_subset_dir/
      group_id/
        videoid_clipid/
          original_frame_name.png
    """
    reset_dir(processed_subset_dir)

    group_dirs = sorted([p for p in extracted_subset_dir.iterdir() if p.is_dir()], key=lambda p: p.name)

    total_videos = 0
    total_clips = 0
    skipped_videos = 0

    for group_dir in group_dirs:
        group_id = group_dir.name
        out_group_dir = processed_subset_dir / group_id
        ensure_dir(out_group_dir)

        video_dirs = sorted([p for p in group_dir.iterdir() if p.is_dir()], key=lambda p: p.name)

        for video_dir in video_dirs:
            total_videos += 1
            video_id = video_dir.name

            frame_files = list_image_files(video_dir)
            n = len(frame_files)

            if n < clip_len:
                skipped_videos += 1
                continue

            clip_ranges = compute_centered_clip_ranges(n, clip_len=clip_len)

            for clip_idx, (s, e) in enumerate(clip_ranges):
                clip_folder_name = f"{video_id}_{clip_idx}"
                out_clip_dir = out_group_dir / clip_folder_name
                ensure_dir(out_clip_dir)

                selected_frames = frame_files[s:e]

                for frame_path in selected_frames:
                    shutil.copy2(frame_path, out_clip_dir / frame_path.name)

                total_clips += 1

    print(f"Subset selesai diproses: {processed_subset_dir}")
    print(f"Total video       : {total_videos}")
    print(f"Total klip 46 frame : {total_clips}")
    print(f"Video dilewati    : {skipped_videos}")

    return {
        "total_videos": total_videos,
        "total_clips": total_clips,
        "skipped_videos": skipped_videos,
    }

## 8. Pipeline lengkap untuk satu subset

In [ ]:
def process_subset_to_46f_dataset(subset_name: str):
    if subset_name not in VALID_SUBSETS:
        raise ValueError(f"Subset tidak valid: {subset_name}")

    class_name = subset_name.split("_")[0]  # real atau fake

    # Folder sementara untuk subset ini
    subset_cache_dir = RAW_CACHE_DIR / subset_name
    subset_extract_dir = RAW_EXTRACT_DIR / subset_name

    reset_dir(subset_cache_dir)
    reset_dir(subset_extract_dir)

    # 1) Download subset
    downloaded_subset_dir = download_subset_from_hf(subset_name, subset_cache_dir)

    # 2) Salin ke area ekstraksi
    move_contents(downloaded_subset_dir, subset_extract_dir)

    # 3) Ekstrak arsip
    extract_all_archives_in_subset(subset_extract_dir)

    # 4) Ratakan folder class internal
    flatten_inner_class_folder(subset_extract_dir, class_name)

    # 5) Bentuk klip 46 frame
    processed_subset_dir = PROCESSED_DATASET_DIR / subset_name
    stats = build_46f_clips_for_subset(
        extracted_subset_dir=subset_extract_dir,
        processed_subset_dir=processed_subset_dir,
        clip_len=CLIP_LEN
    )

    return stats

## 9. Fungsi inspeksi hasil

In [ ]:
def print_tree(root: Path, max_depth: int = 3):
    root = Path(root)
    if not root.exists():
        print(f"Tidak ada folder: {root}")
        return

    def _walk(path: Path, depth: int, prefix: str):
        if depth > max_depth:
            return
        items = sorted(list(path.iterdir()), key=lambda p: (not p.is_dir(), p.name))
        for i, item in enumerate(items):
            connector = "└── " if i == len(items) - 1 else "├── "
            print(prefix + connector + item.name)
            if item.is_dir():
                extension = "    " if i == len(items) - 1 else "│   "
                _walk(item, depth + 1, prefix + extension)

    print(root.name)
    _walk(root, 1, "")


def summarize_processed_root(root: Path):
    root = Path(root)
    if not root.exists():
        print("Root processed belum ada.")
        return

    print("Ringkasan processed root:")
    for subset_dir in sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.name):
        num_groups = 0
        num_clips = 0
        for group_dir in subset_dir.iterdir():
            if group_dir.is_dir():
                num_groups += 1
                num_clips += len([p for p in group_dir.iterdir() if p.is_dir()])
        print(f"- {subset_dir.name}: {num_groups} grup, {num_clips} klip")

In [ ]:
def cleanup_raw_subset(subset_name: str):
    subset_cache_dir = RAW_CACHE_DIR / subset_name
    subset_extract_dir = RAW_EXTRACT_DIR / subset_name

    if subset_cache_dir.exists():
        shutil.rmtree(subset_cache_dir)
        print(f"Dihapus: {subset_cache_dir}")

    if subset_extract_dir.exists():
        shutil.rmtree(subset_extract_dir)
        print(f"Dihapus: {subset_extract_dir}")

## 10. Jalankan subset real_test

In [ ]:
stats_real_test = process_subset_to_46f_dataset("real_test")
print(stats_real_test)

print()
summarize_processed_root(PROCESSED_DATASET_DIR)

print()
print_tree(PROCESSED_DATASET_DIR / "real_test", max_depth=3)
cleanup_raw_subset("real_test")

In [ ]:
version_notes = "Version 1: add processed 46-frame clips for real_test subset"

result = kagglehub.dataset_upload(
    KAGGLE_HANDLE,
    str(PROCESSED_DATASET_DIR),
    version_notes=version_notes
)

print("Upload selesai.")
print(result)

## 11. Jalankan subset real_train

In [ ]:
stats_real_train = process_subset_to_46f_dataset("real_train")
print(stats_real_train)

summarize_processed_root(PROCESSED_DATASET_DIR)
print_tree(PROCESSED_DATASET_DIR / "real_train", max_depth=3)

cleanup_raw_subset("real_train")

In [ ]:
version_notes = "Version 2: add processed 46-frame clips for real_train subset"

result = kagglehub.dataset_upload(
    KAGGLE_HANDLE,
    str(PROCESSED_DATASET_DIR),
    version_notes=version_notes
)

print("Upload selesai.")
print(result)

## 12. Jalankan subset fake_train

In [ ]:
stats_fake_train = process_subset_to_46f_dataset("fake_train")
print(stats_fake_train)

summarize_processed_root(PROCESSED_DATASET_DIR)
print_tree(PROCESSED_DATASET_DIR / "fake_train", max_depth=3)

cleanup_raw_subset("fake_train")

In [ ]:
version_notes = "Version 3: add processed 46-frame clips for fake_train subset"

result = kagglehub.dataset_upload(
    KAGGLE_HANDLE,
    str(PROCESSED_DATASET_DIR),
    version_notes=version_notes
)

print("Upload selesai.")
print(result)

## 13. Jalankan subset fake_test

In [ ]:
stats_fake_test = process_subset_to_46f_dataset("fake_test")
print(stats_fake_test)

summarize_processed_root(PROCESSED_DATASET_DIR)
print_tree(PROCESSED_DATASET_DIR / "fake_test", max_depth=3)

cleanup_raw_subset("fake_test")

In [ ]:
version_notes = "Version 4: add processed 46-frame clips for fake_test subset"

result = kagglehub.dataset_upload(
    KAGGLE_HANDLE,
    str(PROCESSED_DATASET_DIR),
    version_notes=version_notes
)

print("Upload selesai.")
print(result)